In [0]:
from pyspark.sql.functions import col, count, month, year, to_date, concat, lit, when, last_day, lpad
from pyspark.sql import Window
import json
import os

# Paths (Updated to avoid DBFS issues)
output_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/Y02_vacancies_data_2023_24"
checkpoint_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/checkpoints.json"

# Convert DBFS path to local path for JSON handling
local_checkpoint_path = "/dbfs" + checkpoint_path.lstrip("dbfs:")

# Load all files at once (parallel processing)
df = spark.read.parquet("dbfs:/mnt/light/data/world_bank_global_postings/version=3/2025/02/26/00_26_56/*.parquet")

# Filter for US & jobs posted after 2017-01-01
df_us = df.filter((col('nation_name') == "United States") & (col('posted') >= "2023-01-01"))

# Define keywords for solar-related jobs
keywords = [
    "bio polymer", "biobased plastic", "biodegradable plastic", "bioplastic", "compostable plastic", 
    "green plastic", "polylactic acid", "sustainable plastic", "bio bottle", "bio box", "bio pack", 
    "bio pak", "biobased material", "biodegradable packaging", "bio organic", "biopackaging", 
    "compostable packaging", "eco packaging", "ecofriendly packaging", "green packaging", 
    "sustainable packaging", "ghg capture", "carbon capture", "carbon sequestration", "co2 capture", 
    "cogeneration", "combined heat", "combined cycle", "coal gasification", "integrated gasification", 
    "power filtering", "power filters", "flexible ac", "flexible alternating", "statcom", 
    "static synchronous", "reactive power", "series compensation", "shunt compensation", "var compensation", 
    "var compensator", "superconducting electric", "superconducting element", "superconductor", 
    "decentralized energy", "distributed energy", "district energy", "energy storage", "hybrid storage", 
    "power storage", "solar storage", "thermal storage", "highvoltage dc", "noncondensing turbine", 
    "pressure turbine", "trigeneration", "energy recovery", "heat recovery", "load shedding", 
    "peak shaving", "detection sensor", "human detection", "infrared sensor", "pir sensor", 
    "presence detection", "presence sensor", "advanced metering", "ami metering", "net metering", 
    "smart metering", "efficient standby", "power factor", "switched power", "switched mode", 
    "switch mode", "induction cooker", "induction cooking", "induction cooktop", "induction cookware", 
    "induction hob", "induction range", "induction stove", "induction heater", "induction heating", 
    "efficient computing", "lowpower cpu", "lowpower pc", "lowpower processor", "absorption chiller", 
    "absorption cooling", "absorption refrigerator", "air cooled", "free cooling", "double façade", 
    "passive building", "passive design", "passive heating", "passive home", "passive house", 
    "passive solar", "efficient heater", "efficient heating", "efficient bulb", "efficient lighting", 
    "efficient light", "compact fluorescent", "fluorescent lighting", "discharge lamp", "discharge tube", 
    "halogen bulb", "halogen lamp", "halogen light", "tungsten halogen", "emitting diode", "led bulb", 
    "led lamp", "led light", "biodiesel", "bioethanol", "bio-feedstock", "biofuel", "biogas", 
    "biomass", "ethanol", "double glazed", "double glazing", "insulation", "insulator", "green roof", 
    "green roofing", "roof garden", "electric car", "electric drive", "electric engine", "electric jet", 
    "electric motor", "electric propulsion", "electric ship", "electric thruster", "electric vehicle", 
    "charging station", "charging system", "ev charging", "hybrid boat", "hybrid car", "hybrid electric", 
    "hybrid propulsion", "hybrid rocket", "hybrid vehicle", "hydraulic hybrid", "fission energy", 
    "fission reaction", "fission reactor", "fusion energy", "fusion power", "fusion reaction", 
    "fusion reactor", "clean energy", "renewable energy", "gradient power", "halocline", 
    "salinity gradient", "tidal energy", "tidal power", "wave energy", "wave power", "geothermal", 
    "heat exchange", "heat exchanger", "heat pump", "hybrid energy", "hybrid fuel", "hybrid power", 
    "hybrid solar", "hybrid system", "thermal pv", "fuel cell", "hydrogen", "hydro energy", 
    "hydroelectricity", "hydropower", "solar cook", "solar cooker", "solar cooking", "solar furnace", 
    "solar heater", "solar heating", "solar hot water", "solar irrigation", "solar lighting", 
    "solar oven", "solar pond", "solar stove", "solar water", "heat transfer", "photovoltaic cell", 
    "photovoltaic efficiency", "photovoltaic energy", "photovoltaic power", "photovoltaic solar", 
    "pv cell", "pv energy", "pv panel", "pv power", "pv solar", "pv system", "rooftop solar", 
    "solar array", "solar battery", "solar cell", "solar energy", "solar hybrid", "solar panel", 
    "solar power", "solar powered", "solar pump", "solar pv", "solar pv system", "solar thermal", 
    "solar tower", "wind energy", "wind farm", "wind generation", "wind park", "wind power", 
    "wind turbine", "alternative energy", "smart energy", "smart grid", "smart power", 
    "segregated refuse", "waste segregation", "waste sorting", "nuclear power", "nuclear energy", 
    "nuclear generation", "nuclear generated"
]

regex_values = "|".join(keywords)

# Filter for solar-related jobs
df = df_us.filter(col("body").rlike(regex_values))

new_df = (
    df.select("posted", "employment_type", "laa_admin_area_2", "min_years_experience", "isco_08_1")
    .withColumn("month", lpad(month(col("posted")).cast("string"), 2, "0"))
    .withColumn("year", year(col("posted")))
    .groupBy("year", "month", "laa_admin_area_2", "min_years_experience", "isco_08_1")
    .agg(
        count(when(col("employment_type") == 1, True)).alias("full_time_count"),
        count(when(col("employment_type") == 2, True)).alias("part_time_count"),
        count("*").alias("total_count")
    )
    .withColumnRenamed("laa_admin_area_2", "county_id")
    .withColumnRenamed("isco_08_1", "career_type")
    .withColumn("start_date", to_date(concat(col("year"), lit("-"), col("month"), lit("-01")), "yyyy-MM-dd"))
    .withColumn("end_date", last_day(col("start_date")))
)

# Append data as CSV (instead of Parquet)
new_df.write.format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .save(output_path)

In [0]:
# Load processed data
output_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/Y02_vacancies_data_2023_24"

df = spark.read.csv(output_path)

display(df)

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9
year,month,county_id,min_years_experience,career_type,full_time_count,part_time_count,total_count,start_date,end_date
2023,03,US36061,8,1,8,0,8,2023-03-01,2023-03-31
2024,02,US48157,0,3,1,0,1,2024-02-01,2024-02-29
2024,06,US6001,3,1,5,0,5,2024-06-01,2024-06-30
2024,06,US37063,10,2,4,0,4,2024-06-01,2024-06-30
2024,05,US8005,12,2,2,0,2,2024-05-01,2024-05-31
2024,06,US24999,5,2,3,0,4,2024-06-01,2024-06-30
2023,11,US12019,null,1,3,0,3,2023-11-01,2023-11-30
2023,02,US31153,1,3,1,0,1,2023-02-01,2023-02-28
2023,02,US31055,3,2,5,0,5,2023-02-01,2023-02-28
